In [1]:
#MakeMore Part2

In [2]:
#上一节我们用两种方式实现了bigram
#效果不错，但是bigram有个明显的缺陷就是精确度不够
#因为我们的参照太少了，只根据上一个字母来推测下一个字母
#于是乎我们可以考虑添加些上下文？

In [3]:
#但是这又有个很明显的问题
#如果我们用bigram的思路，直接查表
#那么这个表格随着上下文的增长会出现指数级的增长
#比如如果我们用两个字符来决定下一个字母，那么就会出现27*27=729行
#如果用3个会直接变成27*27*27=19683行
#这显然不可接受

In [4]:
#于是本篇我们将学习使用MLP，也就是在micrograd里实现的方法来完成这个模型
#论文已经添加到文件夹下

In [5]:
# andrej这里带我们简单读了下论文
# MLP的本质就是希望模型能够“解释”单词（或者任何语言的单位，在makemore里我们的单位是字符，论文里是单词）
# 我们正常是怎么解释单词的呢？比如说：
# 如果你要给一个人解释“狗dog”这个词
# 我们会说“这和猫一样都是宠物”
# 所以其实我们会发现，语言的“解释”在很大程度上其实是“近义词”的意思
# 就是说我们并不会创造一个新的东西来解释语言
# 而是用语言本身来解释语言
# 所以说MLP的哲学就在于：既然语言本身就能解释语言，那我就去寻找这些语言间的关系
# 所以MLP给每个单词都添加了一个30维的向量，并来微调他们
# 于是乎每一个单词便有了自己在这个30维向量里的位置
# 而我们调整它们之间的位置关系，就可以用距离远近来表示它们之间的关系
# 这就是MLP的基本哲学
# 接下来我们用代码实现这个MLP

In [6]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline
#环境导入，和之前的环境是一样的，前两行是torch，后两行是画图工具

In [7]:
words=open("names.txt","r").read().splitlines()
words[:3]
#还是一样导入words

['emma', 'olivia', 'ava']

In [8]:
len(words)

32033

In [9]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos={i:s for s,i in stoi.items()}
print(itos)
#还是一样建立字符数字表

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [10]:
#接下来我们要像之前一样去建造训练集
#还是一样的：一个xs一个ys一一对应
#但是这次我们要不一样一点：
#我们这次不再是单字节模式了
#也就是说我们推测下一个字符不再只看上一个字符，而是要看更多的东西

block_size=3

X,Y=[],[]

for w in words[:5]:
    
    print(w)
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context),"---->",itos[ix])
        context=context[1:]+[ix]
    print("========")

X=torch.tensor(X)
Y=torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [11]:
#我们可以看到，Part1是字符->字符
#现在是”字符块“->字符，逻辑类似于滚动窗口
#对于前几个字符块左边空的部分用...填充

In [12]:
#并且显然我们是可以自由调整这个字符块的大小的
block_size=10

X,Y=[],[]

for w in words[:5]:
    
    print(w)
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context),"---->",itos[ix])
        context=context[1:]+[ix]
    print("========")

X=torch.tensor(X)
Y=torch.tensor(Y)

emma
.......... ----> e
.........e ----> m
........em ----> m
.......emm ----> a
......emma ----> .
olivia
.......... ----> o
.........o ----> l
........ol ----> i
.......oli ----> v
......oliv ----> i
.....olivi ----> a
....olivia ----> .
ava
.......... ----> a
.........a ----> v
........av ----> a
.......ava ----> .
isabella
.......... ----> i
.........i ----> s
........is ----> a
.......isa ----> b
......isab ----> e
.....isabe ----> l
....isabel ----> l
...isabell ----> a
..isabella ----> .
sophia
.......... ----> s
.........s ----> o
........so ----> p
.......sop ----> h
......soph ----> i
.....sophi ----> a
....sophia ----> .


In [13]:
#现在的字符块就是10，可以看到...变得越来越多
#我们还是采用3作为字符块大小，这样符合论文内容

In [15]:
block_size=3

X,Y=[],[]

for w in words:
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        context=context[1:]+[ix]

X=torch.tensor(X)
Y=torch.tensor(Y)

In [18]:
X.shape,X.dtype,Y.shape,Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [19]:
#可以看到总共有228146个字符块和字符的一一对应